In [1]:
import pandas as pd
from pathlib import Path
import sys
from IPython.display import display, Markdown

In [2]:
sys.path.append("../src")
from explainer import ForensicExplainer

In [3]:
BASE_DIR = Path.cwd().parent
CHATS_CSV = BASE_DIR / "data" / "processed" / "chats_completos.csv"
NODES_FEATURES_CSV = BASE_DIR / "data" / "processed" / "nodos_features.csv"

In [4]:
explainer = ForensicExplainer(model_name="llama-3.3-70b-versatile")

df_chats = pd.read_csv(CHATS_CSV)
df_features = pd.read_csv(NODES_FEATURES_CSV)

In [5]:
# SECCIÓN 1: EXPLICABILIDAD DE GRAFOS (EL BROKER)
display(Markdown("### 🧠 1. Análisis Cualitativo del Perfil 'Broker' (Graph ML)"))

# Buscamos al nodo con el mayor Betweenness Centrality
top_broker = df_features.sort_values(by="Betweenness", ascending=False).iloc[0]['Nodo']

# Filtramos algunos mensajes reales de esta persona para darle contexto al LLM
mensajes_broker = df_chats[df_chats['Emisor'].str.upper().str.contains(str(top_broker).upper(), na=False)]['Mensaje'].dropna().tolist()
# Nos quedamos con una muestra de 5 mensajes largos para que el LLM tenga sustancia
mensajes_muestra = sorted(mensajes_broker, key=len, reverse=True)[:5]

print(f"Analizando perfil de: {top_broker}...")
explicacion_broker = explainer.explain_broker_profile(top_broker, mensajes_muestra)

display(Markdown(f"**Identidad Estructural:** `{top_broker}`"))
display(Markdown(f"**Evaluación Forense (LLM):**\n> {explicacion_broker}"))

### 🧠 1. Análisis Cualitativo del Perfil 'Broker' (Graph ML)

Analizando perfil de: RODOLFO REYES...


**Identidad Estructural:** `RODOLFO REYES`

**Evaluación Forense (LLM):**
> Rodolfo REYES ejerce un rol de intermediario clave en una red de influencias, conectando con figuras como Julio MARTINEZ SOLA y Roberto ROSELLI. Maneja información sobre ayudas públicas y préstamos bancarios, y utiliza su red de contactos para influir en decisiones. Su posición central en la red le permite facilitar comunicaciones y gestiones a alto nivel. Esto sugiere un posible involucramiento en actividades de lobby o tráfico de influencias.

In [6]:
# SECCIÓN 2: EXPLICABILIDAD DE ANOMALÍAS TEMPORALES
display(Markdown("---"))
display(Markdown("### 🚨 2. Análisis Semántico de Anomalías (Isolation Forest)"))

# Vamos a buscar un mensaje que sepamos que es muy tenso/anómalo (por ejemplo, el de la SEPI a las 09:15)
# Buscamos en el dataset base el mensaje que contenga la palabra clandestinamente
df_anomalos = df_chats[df_chats['Mensaje'].str.contains("clandestinamente", na=False, case=False)]

if not df_anomalos.empty:
    evento = df_anomalos.iloc[0]
    fecha = evento['Fecha']
    hora = evento['Hora']
    texto = evento['Mensaje']
    
    print(f"Analizando anomalía detectada el {fecha} a las {hora}...")
    explicacion_anomalia = explainer.explain_anomaly(fecha, hora, texto)
    
    display(Markdown(f"**Evento Anómalo:** `{fecha} {hora} - Longitud: {len(texto)} chars`"))
    display(Markdown(f"**Mensaje Crudo:** _{texto[:150]}..._"))
    display(Markdown(f"**Evaluación Forense (LLM):**\n> {explicacion_anomalia}"))
else:
    print("No se encontró el mensaje anómalo de muestra en el dataset actual.")

---

### 🚨 2. Análisis Semántico de Anomalías (Isolation Forest)

Analizando anomalía detectada el 04/11/2020 a las 9:15:19...


**Evento Anómalo:** `04/11/2020 9:15:19 - Longitud: 1149 chars`

**Mensaje Crudo:** _“Acabo de hablar con el tipo de la SEPI. Estábamos Julio, yo y, de la otra línea, estaba también el contacto, clandestinamente ¿no?... y bueno en teor..._

**Evaluación Forense (LLM):**
> El mensaje analizado refleja una situación de coordinación y presión, ya que se menciona una conversación clandestina con un contacto de la SEPI y se habla de acelerar un proceso para obtener una respuesta antes de fin de año. La mención de "clandestinamente" y la urgencia por obtener una respuesta sugieren una situación de coordinación no transparente. El tono del mensaje es de optimismo cauteloso, pero con un sentido de urgencia. La hora del mensaje, 9:15 am, no es inusual, pero el contenido sí sugiere una situación de presión.